# One-Click Colab Runner

Use this if the whole repo is already in Google Drive.

Manual setup before running:
1. Set Colab runtime to GPU.
2. Put this repo folder in `MyDrive/Neural_Image_Caption_Generator`.
3. Either put Flickr8k in `MyDrive/Neural_Image_Caption_Generator/data/flickr8k`, or set `FLICKR8K_SOURCE_DIR` below to wherever it is in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this only if your repo folder has a different name/location.
REPO_DIR = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')

# If Flickr8k is already inside the repo at data/flickr8k, leave this blank.
# If it is somewhere else in Drive, set it, e.g. Path('/content/drive/MyDrive/flickr8k').
FLICKR8K_SOURCE_DIR = None

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
os.chdir(REPO_DIR)
print('Working in', Path.cwd())

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Prepare Flickr8k

If you do **not** have the dataset in Drive yet, upload your Kaggle `kaggle.json` in the next optional cell and then run the Kaggle download cell.

In [ ]:
# Normal path: dataset is already in Drive.
cmd = ['python', 'scripts/colab_setup.py', '--repo_dir', str(REPO_DIR)]
if FLICKR8K_SOURCE_DIR is not None:
    cmd += ['--source_dir', str(FLICKR8K_SOURCE_DIR)]
print(' '.join(cmd))
import subprocess
subprocess.run(cmd, check=True)

### Optional Kaggle Download Path

Only use these two cells if Flickr8k is not already in Drive. In Kaggle, create an API token and upload `kaggle.json` here.

In [ ]:
# OPTIONAL: upload kaggle.json, then run the next cell.
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPTIONAL: Kaggle download. Uncomment only if you used the kaggle.json cell above.
# !python scripts/colab_setup.py --repo_dir "$REPO_DIR" --download_kaggle

## Validate Code

In [ ]:
!pytest -q

## Train

In [ ]:
!python train.py

## Evaluate

In [ ]:
!python evaluate.py \
  --checkpoint checkpoints/best.pt \
  --data_root data/flickr8k \
  --vocab data/flickr8k/vocab.json \
  --split test \
  --batch_size 64 \
  --results_out results/test_bleu.json

## Visualize Attention

In [ ]:
!python visualize.py \
  --checkpoint checkpoints/best.pt \
  --vocab data/flickr8k/vocab.json \
  --data_root data/flickr8k \
  --split test \
  --num_examples 6 \
  --output_dir results/attention_examples \
  --overlay_style paper \
  --dpi 200

In [ ]:
from IPython.display import Image, display
for path in sorted(Path('results/attention_examples').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))